In [21]:
%reset -f
import sys
for module in list(sys.modules.keys()):
  if module.startswith(("models", "assets", "data", "custom_datasets", "text_classification")):
    del sys.modules[module]
import torch
if torch.cuda.is_available():
  torch.cuda.empty_cache()
!rm -rf /content/*

In [22]:
from torch import nn
import os
from pathlib import Path
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
  from torchinfo import summary
except:
  print("Torchinfo not found! Installing...")
  !pip install -q torchinfo

!git clone https://github.com/asdq11870-cyber/PyTorch
!mv PyTorch/text_classification .
!mv PyTorch/assets .
!mv PyTorch/data/texts .
!mv PyTorch/models .
!mv PyTorch/custom_datasets .
!rm -rf PyTorch

Cloning into 'PyTorch'...
remote: Enumerating objects: 461, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 461 (delta 124), reused 163 (delta 64), pack-reused 210 (from 1)
Receiving objects: 100% (461/461), 27.81 MiB | 23.15 MiB/s, done.
Resolving deltas: 100% (249/249), done.
Filtering content: 100% (4/4), 511.73 MiB | 14.57 MiB/s, done.


In [23]:
from models.NanoGPT import GPT
from text_classification.tokenize import NanoGPTTokenizer
from text_classification import utils, data_setup, engine
tokenizer = NanoGPTTokenizer()
with open("texts/input.txt", "r", encoding="utf-8") as f:
  text = f.read()

tokens = tokenizer.encode(text)
vocab_size = tokenizer.encoder.n_vocab
n = len(tokens)
print(f"Total tokens: {n} | vocab_size: {vocab_size}")
train_tokens = tokens[:int(n*0.9)]
val_tokens = tokens[int(n*0.9):int(n*0.95)]
test_tokens = tokens[int(n*0.95):]

Total tokens: 42548582 | vocab_size: 50257


In [24]:
model0 = GPT(vocab_size=vocab_size, embed_dim=768, heads=12, mlp_dim=3072, mlp_dropout=0.1, attn_dropout=0.1, num_encoder_layers=24,
             context_length=256)
model0 = model0.to(device)
model0 = torch.compile(model0)
batch_size = 8
train_dataloader, val_dataloader, test_dataloader = data_setup.create_dataloaders(train_tokens=train_tokens,
                                                                                  val_tokens=val_tokens,
                                                                                  test_tokens=test_tokens,
                                                                                  context_length=model0.context_length,
                                                                                  batch_size=batch_size)

In [ ]:
writer0 = utils.create_writer("Text_Classification","NanoGPT","5_epochs")
optimiser0 = torch.optim.AdamW(
    params=model0.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1
)
loss_fn0 = nn.CrossEntropyLoss()
scheduler0 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimiser0, T_max=5
)
engine.batch_train(model0, train_dataloader, val_dataloader, test_dataloader, 5, 1, torch.device(device), False, optimiser0, loss_fn0, writer0, scheduler0,
                   "NanoGPT", "saved_models", vocab_size, 8)